In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:

Bank_data1=spark.read.format("delta").load("/Volumes/datanotfound/bronze/raw_data")
Bank_data1.show()

In [0]:
Bank_data2=spark.read.format("delta").load("/Volumes/datanotfound/bronze/raw_data1")
Bank_data2.show()

In [0]:
Bank_data3=spark.read.format("delta").load("/Volumes/datanotfound/bronze/raw_data2")
Bank_data3.display()

In [0]:
Bank_data1.printSchema()
Bank_data2.printSchema()
Bank_data3.printSchema()

In [0]:
print(Bank_data1.count())
print(Bank_data2.count())
print(Bank_data3.count())

In [0]:
Bank_Clean=Bank_data1.dropDuplicates()
Bank_Clean.count()

In [0]:
Bank_Clean1=Bank_data2.dropDuplicates()
Bank_Clean1.count()

In [0]:
Bank_Clean2=Bank_data3.dropDuplicates()
Bank_Clean2.count()

In [0]:
Bank_Clean = Bank_Clean.filter(col("Profit_Margin")>0)
Bank_Clean = Bank_Clean.filter(col("Expenses")>0)
Bank_Clean = Bank_Clean.filter(col("Firm_Revenue")>0)
Bank_Clean.show()
Bank_Clean.count()

In [0]:
Bank_Clean = Bank_Clean.withColumn("City", trim(col("City")))
Bank_Clean = Bank_Clean.withColumn("Region", trim(col("Region")))
Bank_Clean = Bank_Clean.withColumn("Firm_Revenue", trim(col("Firm_Revenue")))
Bank_Clean = Bank_Clean.withColumn("Expenses", trim(col("Expenses")))
Bank_Clean = Bank_Clean.withColumn("Profit_Margin", trim(col("Profit_Margin")))
Bank_Clean.display()

In [0]:
Bank_Clean2.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in Bank_Clean2.columns
]).display()

In [0]:
Bank_Clean=Bank_Clean.fillna({
    "City":"Unknown",
    "Region":"Unknown",
    "Firm_Revenue":0,
    "Expenses":0,
    "Profit_Margin":0
})
Bank_Clean.show()

In [0]:
Bank_Clean1 = Bank_Clean1.filter(col("Customer_ID")>0)
Bank_Clean1 = Bank_Clean1.filter(col("Age")>0)
Bank_Clean = Bank_Clean.filter(col("Branch_ID")>0)

Bank_Clean1.show()
Bank_Clean1.count()

In [0]:
Bank_Clean1.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in Bank_Clean1.columns
]).show()

In [0]:
Bank_Clean1=Bank_Clean1.fillna({
    "Age":0,
    "Customer_Type":"Unknown",
    "City":"Unknown",
    "Region":"Unknown",
    "Bank_Name":"Unknown",
    "Branch_ID":"0"
})
Bank_Clean1.show()
Bank_Clean1.count()

In [0]:
Bank_Clean1 = Bank_Clean1.withColumn("Customer_Type", trim(col("Customer_Type")))
Bank_Clean1 = Bank_Clean1.withColumn("City", trim(col("City")))
Bank_Clean1 = Bank_Clean1.withColumn("Region", trim(col("Region")))
Bank_Clean1 = Bank_Clean1.withColumn("Bank_Name", trim(col("Bank_Name")))

In [0]:
Bank_Clean2=Bank_Clean2.filter(col("Transaction_ID")>0)
Bank_Clean2=Bank_Clean2.filter(col("Customer_ID")>0)
Bank_Clean2=Bank_Clean2.filter(col("Total_Balance")>0)
Bank_Clean2=Bank_Clean2.filter(col("Transaction_Amount")>0)
Bank_Clean2=Bank_Clean2.filter(col("Investment_Amount")>0)
Bank_Clean2.display()
Bank_Clean2.count()


In [0]:
Bank_Clean2 = Bank_Clean2.withColumn("Account_Type", trim(col("Account_Type")))
Bank_Clean2 = Bank_Clean2.withColumn("Investment_Type", trim(col("Investment_Type")))
Bank_Clean2.display()

In [0]:
Bank_Clean2.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in Bank_Clean2.columns
]).display()

In [0]:
from pyspark.sql.window import Window
Bank_Clean2.withColumn("prev_date", lag("Transaction_Date").over(Window.orderBy("Transaction_Date"))) \
  .filter(col("Transaction_Date") < col("prev_date")) \
  .display()

In [0]:
Bank_Clean.write.format("delta").mode("overwrite").save("/Volumes/datanotfound/silver/processed_data")


In [0]:
Bank_Clean1.write.format("delta").mode("overwrite").save("/Volumes/datanotfound/silver/processed_data1")

In [0]:
Bank_Clean2.write.format("delta").mode("overwrite").save("/Volumes/datanotfound/silver/processed_data2")